# Embedding Model Selection — Hands-On

**LLM Engineering · Domain 1 · Roadmap Week 09**

Companion to `02 Literature Notes/LLM Engineering/Embedding Model Selection` and the
deck `Lesson_04_Embedding_Model_Selection.pptx`. Runs fully offline.

**Sources:** MTEB (arXiv:2210.07316); MTEB leaderboard; OpenAI/Cohere/BGE/E5 model cards.

## 0. Setup — two toy 'models' so we can compare selection mechanics offline

In [ ]:
%pip install -q numpy
import numpy as np, re
rng = np.random.RandomState(0)

def make_embedder(dim, noise=0.0):
    "A deterministic hashing embedder; 'noise' simulates a weaker model."
    proj = rng.randn(4096, dim)
    def _embed(texts):
        out = []
        for t in texts:
            base = np.zeros(4096)
            for w in re.findall(r"[a-z0-9]+", t.lower()):
                base[hash(w) % 4096] += 1.0
            v = base @ proj
            if noise: v = v + noise * rng.randn(dim)
            out.append(v)
        v = np.array(out)
        return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-9)
    return _embed

model_A = make_embedder(dim=768, noise=0.0)   # 'strong' model
model_B = make_embedder(dim=256, noise=0.6)   # 'weaker but smaller' model
print("two candidate embedders ready")

## 1. A golden set: queries with known-relevant docs
This is the backbone of model selection — you evaluate on YOUR data, not a benchmark.

In [ ]:
corpus = [
  "Reset your password from the login page.",              # 0
  "Refunds are available within 30 days of purchase.",     # 1
  "Enable two-factor authentication in Security settings.",# 2
  "Support hours are 9am to 5pm, Monday to Friday.",       # 3
  "Recover access to your account via email verification.",# 4
]
queries = [
  ("how do I recover my login?", None),
  ("what is the refund window?", None),
  ("how to turn on 2FA?", None),
]
gold = {
  "how do I recover my login?": {0, 4},
  "what is the refund window?": {1},
  "how to turn on 2FA?": {2},
}
print(len(queries), "queries,", len(corpus), "docs")

## 2. Evaluate candidates: recall@k + MRR

In [ ]:
def evaluate(embed, queries, corpus, gold, k=3):
    C = embed(corpus); Q = embed([q for q,_ in queries])
    hits, rr = 0, []
    for (q,_), qv in zip(queries, Q):
        order = np.argsort(-(C @ qv))[:k]
        ranks = [r for r,idx in enumerate(order,1) if idx in gold[q]]
        if ranks: hits += 1; rr.append(1/ranks[0])
        else: rr.append(0.0)
    return {"recall@%d"%k: round(hits/len(queries),3), "MRR": round(float(np.mean(rr)),3)}

print("model_A (768d):", evaluate(model_A, queries, corpus, gold))
print("model_B (256d):", evaluate(model_B, queries, corpus, gold))

## 3. Add the cost/size axis — the business half of the decision

In [ ]:
def index_cost(n_chunks, dims, bytes_per_dim=4, avg_tokens=400, price_per_m=0.02):
    return {"index_GB": round(n_chunks*dims*bytes_per_dim/1e9, 2),
            "ingest_USD": round(n_chunks*avg_tokens/1e6*price_per_m, 2)}

N = 2_000_000
print("model_A 768d f32 :", index_cost(N, 768, 4))
print("model_B 256d f32 :", index_cost(N, 256, 4))
print("model_A 768d int8:", index_cost(N, 768, 1))     # 4x smaller
print("model_A 96b  bin :", index_cost(N, 768, 0.125)) # 32x smaller

## 4. Put it together: a selection table (quality vs cost)
Pick on the Pareto frontier and document the tradeoff.

In [ ]:
rows = []
for name, embed, dims, b in [
    ("A 768d f32", model_A, 768, 4),
    ("B 256d f32", model_B, 256, 4),
    ("A 768d int8", model_A, 768, 1),
]:
    q = evaluate(embed, queries, corpus, gold)
    c = index_cost(N, dims, b)
    rows.append({"candidate": name, **q, **c})
for r in sorted(rows, key=lambda x: (-x["recall@3"], x["index_GB"])):
    print(r)

> Read it as a Pareto choice: if `A 768d int8` keeps quality while cutting index
size 4×, it likely dominates `A 768d f32`. The offline toy numbers are illustrative;
the *method* is the deliverable.

## 5. Exercises
1. Increase `noise` on model_B until its recall drops below model_A — how much quality does dimensionality buy here?
2. Add a 4th candidate at 128d and re-rank the selection table.
3. Compute the monthly query cost (not just ingest) assuming 500k queries/month.
4. Replace the toy embedders with `sentence-transformers` models and a real golden set from your corpus.
5. Write the one-paragraph recommendation you'd put in the selection memo.

## Links
- Literature note: `02 Literature Notes/LLM Engineering/Embedding Model Selection`
- Snippet: `04 Code Snippets/LLM/Comparing Embedding Models on a Golden Set`
- MOC: `06 Maps of Content/LLM Engineering Concepts`